In [1]:
import os
import json
import random
import yfinance as yf
import pandas as pd
from time import sleep
from symbols import my_picks
from datetime import datetime

os.makedirs('data/stock_info', exist_ok=True)

csv_path = 'data/stocks-revenue-growth.csv'
try:
    prev_df = pd.read_csv(csv_path)
    prev_data = {r['symbol']: (round(float(r['revenue growth']), 4), str(r['date'])) for _, r in prev_df.iterrows()}
except Exception:
    prev_data = {}

rows = []
for symbol in my_picks:
    info_path = f'data/stock_info/{symbol}.json'
    try:
        with open(info_path) as f:
            cached_info = json.load(f)
    except Exception:
        cached_info = None

    if cached_info and cached_info['quoteType'] != 'EQUITY':
        continue

    sleep(random.uniform(1, 2))  # Avoid hitting API rate limits
    stock = yf.Ticker(symbol)
    info = stock.info

    with open(info_path, 'w') as f:
        json.dump(info, f, indent=2)

    if info['quoteType'] != 'EQUITY':
        print(f"Skipping {symbol}, not an equity symbol")
        continue

    print(f"Fetching revenue for {symbol}...")
    qis = stock.quarterly_income_stmt
    if 'Total Revenue' not in qis.index:
        print(f"  no Total Revenue row for {symbol}")
        continue
    rev = qis.loc['Total Revenue'].dropna().sort_index(ascending=False)
    if len(rev) < 5:
        print(f"  need 5 quarters, got {len(rev)} for {symbol}")
        continue
    rev_growth = round(float(rev.iloc[0] / rev.iloc[4] - 1), 4)
    quarter_end = rev.index[0].date()

    today = datetime.now().date()
    report_date = quarter_end
    try:
        ed = stock.earnings_dates
        if ed is not None and len(ed):
            past = [d.date() for d in ed.index if d.date() <= today and d.date() >= quarter_end]
            if past:
                report_date = min(past)
    except Exception:
        pass

    next_earnings = None
    try:
        cal = stock.calendar or {}
        ed_cal = cal.get('Earnings Date') or []
        upcoming = [d for d in ed_cal if d >= today]
        if upcoming:
            next_earnings = min(upcoming)
    except Exception:
        pass

    if symbol in prev_data and (rev_growth, str(report_date)) != prev_data[symbol]:
        old_rev, old_date = prev_data[symbol]
        print(f"  {old_rev:.2%} ({old_date}) => {rev_growth:.2%} ({report_date})")

    rows.append({
        'symbol': symbol,
        'revenue growth': rev_growth,
        'date': report_date,
        'next earnings': next_earnings,
    })

print(f'Total: {len(rows)} stocks')
df = pd.DataFrame(rows)
df.to_csv(csv_path, index=False)

Fetching revenue for BRK-B...
Fetching revenue for PL...
Fetching revenue for RKLB...
Fetching revenue for NVDA...
Fetching revenue for AVGO...
Fetching revenue for TSLA...
Fetching revenue for MSFT...
Fetching revenue for GOOG...
Fetching revenue for AAPL...
Fetching revenue for AMZN...
Fetching revenue for META...
Fetching revenue for NFLX...
Fetching revenue for SHOP...
Fetching revenue for NET...
Fetching revenue for PLTR...
Fetching revenue for BKNG...
Fetching revenue for ISRG...
Total: 17 stocks


In [2]:
df = df.sort_values('revenue growth', ascending=False)
for col in ['revenue growth']:
    df[col] = df[col].map('{:.2%}'.format)

display(df)

,symbol,revenue growth,date,next earnings
3,NVDA,85.23%,2026-05-20,2026-08-26
14,PLTR,84.71%,2026-05-04,2026-08-03
2,RKLB,63.46%,2026-05-07,2026-08-06
1,PL,41.05%,2026-03-19,None
12,SHOP,34.32%,2026-05-05,2026-08-05
13,NET,33.54%,2026-05-07,2026-07-30
10,META,33.08%,2026-04-29,2026-07-29
4,AVGO,29.47%,2026-03-04,2026-09-03
16,ISRG,22.96%,2026-04-21,2026-07-21
7,GOOG,21.79%,2026-04-29,2026-07-23
